# Graph Neural Networks (GNNs)

## Core Idea: Message Passing

Nodes iteratively update their representation by aggregating neighbor information.

$$h_i^{(l+1)} = \text{UPDATE}\Big(h_i^{(l)},\; \text{AGG}\big(\{h_j^{(l)} : j \in \mathcal{N}(i)\}\big)\Big)$$

Each round: **Message** (neighbors send features) → **Aggregate** (sum/mean/max) → **Update** (linear + nonlinearity). After $L$ layers, each node encodes its $L$-hop neighborhood.

## GCN Layer

$$h_i^{(l+1)} = \text{ReLU}\Big(W^{(l)} \sum_{j \in \mathcal{N}(i) \cup \{i\}} \frac{h_j^{(l)}}{\sqrt{|\mathcal{N}(i)| \cdot |\mathcal{N}(j)|}}\Big)$$

Normalized neighbor averaging + learnable weights + self-loop.



### Basic Implementation

In [256]:
import torch
import torch.nn.functional as F

In [257]:
# Nodes w/ 3 feature columns
X = torch.tensor([
    [1.0, 0.0, 0.0],  # node 0
    [0.0, 1.0, 0.0],  # node 1
    [0.0, 0.0, 1.0],  # node 2
    [1.0, 1.0, 0.0],  # node 3
    [0.0, 1.0, 1.0],  # node 4
])

# Adjacency matrix
A = torch.tensor([
    [0, 1, 0, 1, 0],
    [1, 0, 1, 0, 1],
    [0, 1, 0, 0, 1],
    [1, 0, 0, 0, 1],
    [0, 1, 1, 1, 0],
], dtype=torch.float)

In [258]:
# Make every node its own neighbor as well (update from its own value)
I = torch.eye(5)
A_hat = A + I

In [259]:
# Builds a degree matrix, which is a diagonal matrix where D[i][i] represents the number of connections the i-th node has (including itself)
# Sums across the columns of the adjacency matrix
D = torch.diag(A_hat.sum(dim=1)) # builds diagonal matrix from 1D tensor

In [260]:
# Symmetric normalization step within the message passing / update mechanism
D_inv_sqrt = torch.diag(1.0 / D.diag().sqrt())

# The left multiply transforms each entry per row into 1/sqrt(d_i) * A_hat[i,j]
# The right multiply transforms each entry per col into 1/sqrt(d_j) * A_hat[i,j]
A_norm = D_inv_sqrt @ A_hat @ D_inv_sqrt

In [261]:
# Apply the GCN formula
torch.manual_seed(1)
W = torch.randn(3, 2, requires_grad=True)

H = F.relu(A_norm @ X @ W)

print("Input features X:\n", X)
print("\nNormalized adjacency A_norm:\n", A_norm.round(decimals=3))
print("\nOutput node embeddings H:\n", H.round(decimals=3))

Input features X:
 tensor([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.],
        [1., 1., 0.],
        [0., 1., 1.]])

Normalized adjacency A_norm:
 tensor([[0.3330, 0.2890, 0.0000, 0.3330, 0.0000],
        [0.2890, 0.2500, 0.2890, 0.0000, 0.2500],
        [0.0000, 0.2890, 0.3330, 0.0000, 0.2890],
        [0.3330, 0.0000, 0.0000, 0.3330, 0.2890],
        [0.0000, 0.2500, 0.2890, 0.2890, 0.2500]])

Output node embeddings H:
 tensor([[0.4790, 0.5640],
        [0.0000, 0.2980],
        [0.0000, 0.2550],
        [0.3490, 0.5160],
        [0.0000, 0.4780]], grad_fn=<RoundBackward1>)


### Example

Consider two communities of 30 people each. Within each community, there is a 40% chance of connection, and a 5% chance of connection across communities.

Those in community 0 tends to have features centered around [1, 0] and community 1 tends to have features centered around [0, 1].

We want to classify which communities individuals belong in based off of their features. However, there is a huge amount of noise (1.5) relative to the signal (1).

In [262]:
import random
import math

In [263]:
# 0/1 label indicates community
n_per_class = 30
n_nodes = 2 * n_per_class
labels = torch.tensor([0] * n_per_class + [1] * n_per_class) # [0, 0..., 1]

# We use fancy indexing to have centers become shape (60, 2) -> elementwise addition with standard noise * multiplier
noise = 1.5
centers = torch.tensor([
    [1, 0],
    [0, 1]
], dtype=torch.float32)
features = centers[labels] + torch.randn(n_nodes, 2) * noise

In [264]:
# Build adjacency matrix
p_same_comm = 0.20
p_diff_comm = 0.05

same_community = (labels.unsqueeze(0) == labels.unsqueeze(1)) # (n, n)
probs = torch.where(same_community, p_same_comm, p_diff_comm)
upper = torch.triu(torch.bernoulli(probs), diagonal=1) # take upper triangular of torch.bernoulli(), where each element of that tensor is independently sampled as 0/1 based on input probs

A = upper + upper.T

Try classifying based solely on features

In [265]:
feature_pred = (features[:, 1] > features[:, 0]).to(torch.int64)
feature_acc = (feature_pred == labels).float().mean()
print(f"The feature-only predictions had {feature_acc * 100:.2f}% accuracy")

The feature-only predictions had 63.33% accuracy


Try classifying using GCN

In [266]:
# Define neural network layer and architecture

def gcn_normalize(A: torch.Tensor) -> torch.Tensor:
    """Compute and return D^{-1/2} @ (A + I) @ D^{-1/2}"""
    A_hat = A + torch.eye(A.size(0))
    D = A_hat.sum(dim=1)
    D_inv_sqrt = torch.diag(1.0 / D.sqrt())

    return D_inv_sqrt @ A_hat @ D_inv_sqrt

class GCNLayer(torch.nn.Module):
    def __init__(self, in_dim: int, out_dim: int) -> None:
        super().__init__()
        # Create learnable weight matrix
        # Wrap in torch.nn.Parameter() so that optimizer updates it during training
        self.W = torch.nn.Parameter(torch.empty(in_dim, out_dim))
        # Initializes weights using specific initialization strategy
        torch.nn.init.kaiming_uniform_(self.W, a=math.sqrt(5))
    
    def forward(self, A_norm: torch.Tensor, H: torch.Tensor) -> torch.Tensor:
        """
        Args:
            A_norm (N, N): Normalized adjacency weight matrix based on number of neighbors
            H (N, in_dim): Node features
        
        Returns:
            A_norm @ H @ self.W
        """
        return A_norm @ H @ self.W
    
class GCN(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim) -> None:
        super().__init__()
        self.layer1 = GCNLayer(in_dim, hidden_dim)
        self.layer2 = GCNLayer(hidden_dim, out_dim)
    
    def forward(self, A_norm: torch.Tensor, X: torch.Tensor) -> torch.Tensor:
        H = F.relu(self.layer1(A_norm, X))
        H = self.layer2(A_norm, H)
        return H

In [267]:
# Create stratified train/test splits

A_norm = gcn_normalize(A)

train_ratio = 0.75
per_class_rank = torch.zeros(n_nodes, dtype=torch.long)
class_sizes = torch.zeros(2, dtype=torch.long)

for c in range(2):
    mask = (labels == c)
    class_sizes[c] = mask.sum()

    # torch.randperm(n) returns a random permutation of integers 0...n-1
    # Then, each node in a specific class is given a "rank" position
    # That way the first 75% of each class can be used for train/test
    per_class_rank[mask] = torch.randperm(int(class_sizes[c]))

# Masks to choose which nodes (in both classes) to keep in train/test
train_mask = per_class_rank < (class_sizes[labels] * train_ratio).long()
test_mask = ~train_mask

In [268]:
# Instantiate the model and optimizer

# in_dim=2 for 2 features, hidden_dim=16 as an empirical hyperparameter, out_dim=2 for 2 output classes
model = GCN(in_dim=2, hidden_dim=16, out_dim=2)

# Weight decay is a regularization term, a component added in the loss to penalize large weights
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

In [269]:
# Train the model

for epoch in range(300):
    # Sets model in "training mode"
    model.train()
    # model.forward()
    logits = model(A_norm, features)
    # Compute a scalar number representing how wrong predictions were
    loss = F.cross_entropy(logits[train_mask], labels[train_mask])
    # Compute gradients for dL_dw for each parameter w
    loss.backward()
    # Optimize parameters using gradients
    optimizer.step()
    # Clear gradients to avoid accumulation
    optimizer.zero_grad()

    if (epoch + 1) % 50 == 0:
        # Sets model out of "training mode"
        model.eval()
        with torch.no_grad(): # skip building computational graph
            pred: torch.Tensor = model(A_norm, features).argmax(dim=1)
            train_acc = (pred[train_mask] == labels[train_mask]).float().mean()
            test_acc = (pred[test_mask] == labels[test_mask]).float().mean()
        print(f'Epoch {epoch + 1}: Loss of {loss:.4f}')
        print(f'Train: {train_acc:.2f} | Test: {test_acc:.2f}')


Epoch 50: Loss of 0.4936
Train: 0.82 | Test: 1.00
Epoch 100: Loss of 0.4135
Train: 0.80 | Test: 0.94
Epoch 150: Loss of 0.3676
Train: 0.84 | Test: 0.88
Epoch 200: Loss of 0.3535
Train: 0.84 | Test: 0.88
Epoch 250: Loss of 0.3467
Train: 0.86 | Test: 0.88
Epoch 300: Loss of 0.3427
Train: 0.86 | Test: 0.88


Try applying the trained GCN to a larger graph

In [270]:
# Obtain the inputs for the model (A_norm and features)

n_per_class = 2000
n_nodes = n_per_class * 2
labels = torch.tensor([0] * n_per_class + [1] * n_per_class)

centers = torch.Tensor([
    [1, 0],
    [0, 1]
])
features = torch.randn(n_nodes, 2) * noise + centers[labels]

same_community = (labels.unsqueeze(0) == labels.unsqueeze(1))
probs = torch.where(same_community, p_same_comm, p_diff_comm)
upper = torch.triu(torch.bernoulli(probs))
A = upper + upper.T
A_norm = gcn_normalize(A)

In [271]:
# # Stratify train/test

# train_ratio = 0.75
# per_class_rank = torch.empty(n_nodes, dtype=torch.int64)
# class_sizes = torch.empty(2, dtype=torch.int64)

# for c in range(2):
#     mask = (labels == c)
#     class_sizes[c] = mask.sum()
#     per_class_rank[mask] = torch.randperm(int(class_sizes[c]))

# train_mask = per_class_rank < (class_sizes[labels] * train_ratio).long()
# test_mask = ~train_mask


In [277]:
# Evaluate the model

model.eval()
with torch.no_grad():
    pred: torch.Tensor = model(A_norm, features)
    pred = pred.argmax(dim=1) # argmax on logits works just like taking class with highest probability
    print(pred.shape)
    print(labels.shape)
    acc = (pred == labels).float().mean()
    print(f'{acc * 100:.2f}%')

torch.Size([4000])
torch.Size([4000])
100.00%


The output is actually significantly better when applied to a larger graph due to a greater number of neighbors for each node (given probabilities) and noise cancels out by the law of large numbers.